<a href="https://colab.research.google.com/github/ext-colorful/LangChain-Essentials/blob/main/%F0%9F%A7%AACreate_Agent%F0%9F%A7%AAA2_real_world_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[Quickstart 快速入门===>Build a real-world agent  构建一个现实世界的代理](https://docs.langchain.com/oss/python/langchain/quickstart#build-a-real-world-agent)

### Next, build a practical weather forecasting agent that demonstrates key production concepts:
接下来，构建一个实用的天气预报代理程序，以演示关键的生产概念：
1. Detailed system prompts for better agent behavior
详细的系统提示有助于改善代理行为
2. Create tools that integrate with external data
创建可与外部数据集成的工具
3. Model configuration for consistent responses
一致响应的模型配置
4. Structured output for predictable results
结构化输出， 结果可预测
5. Conversational memory for chat-like interactions
用于类似聊天互动的对话记忆
6. Create and run the agent create a fully functional agent
创建并运行代理 ，创建一个功能齐全的代理

### Let’s walk through each step:
让我们一步一步来：

In [1]:
!pip install langchain==1.0.7 langgraph==1.0.3 langchain-openai==1.0.3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.7/93.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.8/156.8 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.5/82.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 473.8/473.8 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.3/208.3 kB 10.9 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.80
    Uninstalling langchain-core-0.3.80:
      Successfully uninstalled langchain-core-0.3.80
  Attempting uninstall: langchain
    Found existing installation: langchain 0.3.27
    Uninstalling langchain-0.3.27:
      Successfully uninstalled langchain-0.3.27


In [4]:
# Used to securely store your API key
from google.colab import userdata
import os

# Retrieve the API key from Colab secrets
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["OPENAI_API_BASE"] = userdata.get('OPENAI_API_BASE');

#### 1. Define the system prompt  定义系统提示符

The system prompt defines your agent’s role and behavior. Keep it specific and actionable:

系统提示定义了代理的角色和行为。请确保提示内容具体明确且可操作：

```
SYSTEM_PROMPT = """你是一名擅长使用双关语的专业天气预报员。

你可以使用两个工具：

get_weather_for_location：用来获取指定地点的天气。
get_user_location：用来获取用户所在的位置。

如果用户询问天气，你需要确保你知道地点。如果从问题内容判断出用户指的是“他们所在的位置”，则使用 get_user_location 工具来确定位置。"""
```

In [5]:
SYSTEM_PROMPT = """You are an expert weather forecaster, who speaks in puns.

You have access to two tools:

- get_weather_for_location: use this to get the weather for a specific location
- get_user_location: use this to get the user's location

If a user asks you for the weather, make sure you know the location. If you can tell from the question that they mean wherever they are, use the get_user_location tool to find their location."""

#### 2. Create tools  创建工具

Tools let a model interact with external systems by calling functions you define. Tools can depend on runtime context and also interact with agent memory.

工具允许模型通过调用您定义的函数与外部系统交互。工具可以依赖于运行时上下文 ，也可以与代理的内存进行交互。

Notice below how the get_user_location tool uses runtime context:

请注意以下 get_user_location 工具如何使用运行时上下文：

In [6]:
from dataclasses import dataclass
from langchain.tools import tool, ToolRuntime

@tool
def get_weather_for_location(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

@dataclass
class Context:
    """Custom runtime context schema."""
    user_id: str

@tool
def get_user_location(runtime: ToolRuntime[Context]) -> str:
    """Retrieve user information based on user ID."""
    user_id = runtime.context.user_id
    return "Florida" if user_id == "1" else "SF"

#### 3. Configure your model  配置您的模型

Set up your language model with the right parameters for your use case:

根据您的使用场景，设置合适的语言模型参数 ：

In [7]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "openai:gpt-5-nano",
    temperature=0.5,
    timeout=10,
    max_tokens=1000
)

In [12]:
print(model)

client=<openai.resources.chat.completions.completions.Completions object at 0x7aec88ae9ca0> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x7aec85f14d10> root_client=<openai.OpenAI object at 0x7aec871db860> root_async_client=<openai.AsyncOpenAI object at 0x7aec871db4d0> model_name='gpt-5-nano' model_kwargs={} openai_api_key=SecretStr('**********') openai_api_base='https://api.chatanywhere.org' request_timeout=10.0 max_tokens=1000


#### 4. Define response format  定义响应格式

Optionally, define a structured response format if you need the agent responses to match a specific schema.

如果您需要代理响应与特定模式匹配，则可以选择定义结构化响应格式。

In [8]:
# We use a dataclass here, but Pydantic models are also supported.
# 我们在这里使用数据类，但 Pydantic 模型也受支持。
@dataclass
class ResponseFormat:
    """Response schema for the agent."""
    # A punny response (always required) 一个双关语式的回答（必须包含）
    punny_response: str
    # Any interesting information about the weather if available 如果可以，提供一些有趣的天气信息
    weather_conditions: str | None = None

#### 5. Add memory  添加内存

Add memory to your agent to maintain state across interactions. This allows the agent to remember previous conversations and context.

为智能体添加记忆功能 ，以便在交互过程中保持状态。这样，智能体就能记住之前的对话和上下文。

In [9]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

> ⚠️ In production, use a persistent checkpointer that saves to a database. See [Add and manage memory](https://docs.langchain.com/oss/python/langgraph/add-memory#manage-short-term-memory) for more details.
在生产环境中，请使用将数据保存到数据库的持久化检查点。有关更多详细信息，请参阅 “添加和管理内存” 。

#### 6. Create and run the agent
创建并运行代理

Now assemble your agent with all the components and run it!

现在将所有组件组装到您的代理中并运行它！

In [10]:
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy

agent = create_agent(
    model=model,
    system_prompt=SYSTEM_PROMPT,
    tools=[get_user_location, get_weather_for_location],
    context_schema=Context,
    response_format=ToolStrategy(ResponseFormat),
    checkpointer=checkpointer
)

# `thread_id` is a unique identifier for a given conversation.
# `thread_id` 是给定对话的唯一标识符。
config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather outside?"}]},
    # what is the weather outside? 外面的天气怎么样？
    config=config,
    context=Context(user_id="1")
)

print(response['structured_response'])
# ResponseFormat(
#     punny_response="Florida is still having a 'sun-derful' day! The sunshine is playing 'ray-dio' hits all day long! I'd say it's the perfect weather for some 'solar-bration'! If you were hoping for rain, I'm afraid that idea is all 'washed up' - the forecast remains 'clear-ly' brilliant!",
#     punny_response：佛罗里达依然是个“阳光妙不可言”的好日子！阳光整天都在播放它的“光芒电台”金曲！我得说，这天气简直是为“太阳庆典”量身定做的！如果你期待下雨，那恐怕这个念头已经被“冲刷”掉啦——预报依然“一片晴明”！
#     weather_conditions="It's always sunny in Florida!"
#     weather_conditions：佛罗里达永远阳光灿烂！
# )

# Note that we can continue the conversation using the same `thread_id`.
# 请注意，我们可以使用相同的 `thread_id` 继续对话。
response = agent.invoke(
    {"messages": [{"role": "user", "content": "thank you!"}]},
    config=config,
    context=Context(user_id="1")
)

print(response['structured_response'])
# ResponseFormat(
#     punny_response="You're 'thund-erfully' welcome! It's always a 'breeze' to help you stay 'current' with the weather. I'm just 'cloud'-ing around waiting to 'shower' you with more forecasts whenever you need them. Have a 'sun-sational' day in the Florida sunshine!",
#     punny_response：你真是“雷”厉风行地客气呀！帮你掌握天气动态对我来说一直是件“小风”得意的事。我就在这里“云”游四方，随时准备“雨”你所需，给你送去更多天气预报。祝你在佛罗里达的阳光里度过一个“阳”光璀璨的好日子！
#     weather_conditions=None
# )

RateLimitError: Error code: 429 - {'error': {'message': 'gpt-5-nano-ca模型免费API限制每日200次请求，请00:00后再试，如有更多需求，请访问 https://api.chatanywhere.tech/#/shop 购买付费API。The free account is limited to 200 requests per day. Please try again after 00:00 the next day. If you have additional requirements, please visit https://api.chatanywhere.tech/#/shop to purchase a premium key.(当前请求使用的ApiKey: sk-few****KShv)【如果您遇到问题，欢迎加入QQ群咨询：1048463714】', 'type': 'chatanywhere_error', 'param': None, 'code': '429 TOO_MANY_REQUESTS'}}

In [10]:
print(response['structured_response'].punny_response)

You're very welcome! If you need any more weather puns or forecasts, I'm always here to make your day a little brighter. Stay sun-sational!
